# 01 · Ingesta, limpieza y perfil de compra (F1 y F2)

**ObservaCompras IA** · MVP · Aplicaciones de Inteligencia Artificial (MTI-493), UTFSM  
Profesor: Sr. Werner Creixell · Alumnos: Alejandra Cornejo Jara y Leoncio Cornejo Miranda

Este cuaderno implementa las funcionalidades **F1** (ingesta y limpieza) y **F2** (perfil de compra con 10 variables por organismo y año) de la Tabla 2 del documento.

**Datos de entrada.** Descargue desde https://datos-abiertos.chilecompra.cl los archivos del sector Salud (órdenes de compra y licitaciones) de 2023 y 2024 y copie los `.7z`, `.zip` o `.csv` en `MyDrive/observacompras-ia/data/raw/`. La ingesta los descomprime y reconoce cada archivo por su encabezado.

In [ ]:
# Preparación del entorno (Google Colab o local)
import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # El repositorio se clona en Drive para que datos y modelos persistan entre cuadernos
    RAIZ = "/content/drive/MyDrive/observacompras-ia"
    if not os.path.exists(RAIZ):
        subprocess.run(["git", "clone", "https://github.com/lcornejom/observacompras-ia", RAIZ], check=True)
    os.chdir(RAIZ)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(RAIZ)
sys.path.insert(0, os.path.join(RAIZ, "src"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Modo de periodos: "anio" (2023 y 2024 completos, como en el documento) o
# "demo" (solo un semestre: 1er trimestre = "2023", 2º trimestre = "2024")
MODO = "anio"

from observacompras import config as C
print("Raíz del proyecto:", RAIZ)

In [ ]:
import os
print(sorted(os.listdir(C.DIR_RAW)))

## F1 · Ingesta y limpieza

Pasos: filtrar el sector Salud y 2023–2024; colapsar las líneas de ítem a una fila por orden de compra; excluir OC canceladas; validar montos (> 0) y fechas; normalizar RUT de organismos y proveedores; y anonimizar a los proveedores persona natural (RUT < 50.000.000) con un seudónimo estable, conforme a la Ley 19.628 y la Ley 21.719 (Tabla 5).

In [ ]:
from observacompras import ingesta
import json
resumen = ingesta.ejecutar(C.DIR_RAW, modo=MODO)
print(json.dumps(resumen, indent=2, ensure_ascii=False))

In [ ]:
import pandas as pd
from observacompras.ingesta import leer_oc
oc = leer_oc()
lic = pd.read_parquet(C.DIR_PROCESADOS / 'licitaciones.parquet')
display(oc.head())
display(lic.head())

## F2 · Perfil de compra

| Variable | Definición |
|---|---|
| N.º de compras | Órdenes de compra emitidas |
| Monto total (log) | ln(1 + suma del monto neto en CLP) |
| N.º de proveedores | Proveedores distintos por RUT |
| HHI | Σ (participación % de cada proveedor)²; máximo 10.000 |
| CR1 | Participación del principal proveedor en el monto |
| % trato directo | Fracción de OC por trato directo |
| Monto medio (log) | ln(1 + monto medio por OC) |
| % proveedores nuevos | Proveedores cuya primera OC con el organismo en el año cae en el 2º semestre |
| N.º de rubros | Rubros (nivel 1) distintos comprados |
| % con un solo oferente | Licitaciones adjudicadas con un único oferente |

In [ ]:
from observacompras import perfil
perfiles = perfil.ejecutar()
display(perfiles.groupby('anio').size())
display(perfiles[C.VARIABLES].describe().T.round(3))

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 5, figsize=(16, 5))
for ax, v in zip(axs.ravel(), C.VARIABLES):
    for a in C.ANIOS:
        ax.hist(perfiles.loc[perfiles.anio == a, v], bins=30, alpha=.6, label=str(a))
    ax.set_title(C.NOMBRES_VARIABLES[v], fontsize=9)
axs[0, 0].legend(); plt.tight_layout(); plt.show()